# Anomaly Detection

*Generated by DoML — Do Machine Learning*

In [ ]:
# Reproducibility — REPR-01 (non-negotiable per CLAUDE.md)
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import json
from pathlib import Path
from IPython.display import Markdown, display

# Project root — REPR-02 (non-negotiable per CLAUDE.md)
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/home/jovyan/work'))

with open(PROJECT_ROOT / '.planning' / 'config.json') as f:
    config = json.load(f)

print(f"Project root:  {PROJECT_ROOT}")
print(f"Problem type:  {config.get('problem_type', 'unknown')}")
print(f"Dataset path:  {config.get('dataset', {}).get('path', 'data/raw/')}")

## 1. Data Load

Load input dataset for anomaly detection. Default input is `data/raw/` as configured in
`config.json`; override with `--file path/to/file` when invoking the skill.

Only numeric columns are used for anomaly detection algorithms. Categorical columns are
listed for reference but excluded from scoring.

In [ ]:
import duckdb
import pandas as pd

dataset_path = config.get('dataset', {}).get('path', 'data/raw/')
input_path = PROJECT_ROOT / dataset_path

SUPPORTED = {'.csv', '.parquet', '.xlsx'}
if input_path.is_dir():
    files = sorted([f for f in input_path.glob('**/*') if f.is_file() and f.suffix.lower() in SUPPORTED])
    if not files:
        raise FileNotFoundError(f"No supported files found in {input_path}")
    input_file = files[0]
    if len(files) > 1:
        print(f"Multiple files found — using first: {input_file.name}")
        print(f"Use --file to specify a different file.")
else:
    input_file = input_path

ext = input_file.suffix.lower()
con = duckdb.connect()

if ext == '.csv':
    df_full = con.execute(f"SELECT * FROM read_csv_auto('{input_file}')").df()
elif ext == '.parquet':
    df_full = con.execute(f"SELECT * FROM read_parquet('{input_file}')").df()
elif ext == '.xlsx':
    df_full = pd.read_excel(input_file)
else:
    raise ValueError(f"Unsupported file format: {ext}")

con.close()

numeric_cols = df_full.select_dtypes(include='number').columns.tolist()
cat_cols = df_full.select_dtypes(exclude='number').columns.tolist()

print(f"Input file:      {input_file.name}")
print(f"Shape:           {len(df_full):,} rows × {len(df_full.columns)} columns")
print(f"Numeric columns: {numeric_cols}")
if cat_cols:
    print(f"Categorical columns (excluded from scoring): {cat_cols}")

if not numeric_cols:
    raise ValueError("No numeric columns found. Anomaly detection requires at least one numeric column.")

# Drop rows where ALL numeric columns are null
df = df_full[numeric_cols].dropna(how='all').reset_index(drop=True)
print(f"\nRows used for scoring: {len(df):,} (dropped {len(df_full) - len(df)} all-null rows)")

## 2. Tidy Data Validation

<!-- ANOM-02: tidy validation before analysis -->
Validates data against tidy principles before running any anomaly algorithm.
Violations are **flagged only** — not silently fixed.

In [ ]:
display(Markdown("### Tidy Validation"))
issues = []

# Check 1: Duplicate rows
dup_count = df_full.duplicated().sum()
if dup_count > 0:
    display(Markdown(f"**[VIOLATION] Duplicate rows: {dup_count} exact duplicates**"))
    display(df_full[df_full.duplicated(keep=False)].head(3))
    issues.append(f"Duplicate rows ({dup_count}) — consider df.drop_duplicates() before modelling")
else:
    print("✓ No duplicate rows")

# Check 2: Multi-value cells in string columns
multi_cols = []
for col in df_full.select_dtypes(include='object').columns:
    non_null = df_full[col].dropna()
    if len(non_null) == 0:
        continue
    pct = non_null.str.contains(r'[,;|]', regex=True, na=False).mean()
    if pct > 0.05:
        multi_cols.append({'column': col, 'pct_multi_value': round(pct * 100, 1)})
if multi_cols:
    display(Markdown("**[VIOLATION] Multi-value cells (>5% contain comma/semicolon/pipe):**"))
    display(pd.DataFrame(multi_cols))
    issues.append(f"Multi-value columns: {[r['column'] for r in multi_cols]}")
else:
    print("✓ No multi-value cells detected")

# Check 3: Mixed entity types (column-name prefix heuristic)
prefixes = {}
for col in df_full.columns:
    prefix = col.split('_')[0].lower() if '_' in col else col[:4].lower()
    prefixes[prefix] = prefixes.get(prefix, 0) + 1
dominant = {p for p, c in prefixes.items() if c >= 2}
if len(dominant) >= 3:
    display(Markdown(
        f"**[POTENTIAL VIOLATION] Mixed entity types:** {len(dominant)} column-name prefixes — "
        f"`{'`, `'.join(sorted(dominant)[:5])}`"
    ))
    issues.append("Mixed entity column prefixes — verify table does not combine multiple entities")
else:
    print("✓ No obvious mixed entity types")

if issues:
    display(Markdown("\n**Tidy violations — review before proceeding:**"))
    for iss in issues:
        print(f"  • {iss}")
else:
    print("\n✓ All tidy checks passed")

## 3. Isolation Forest

<!-- ANOM-01 -->
Isolation Forest isolates observations by randomly partitioning features.
Anomalous points require fewer splits to isolate — they have lower (more negative) anomaly scores.

Parameters: `contamination='auto'`, `n_estimators=100`, `random_state=SEED`

In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(df.fillna(df.median()))

iso_forest = IsolationForest(
    contamination='auto',
    n_estimators=100,
    random_state=SEED,
    n_jobs=-1
)
iso_forest.fit(X)

# score_samples returns negated anomaly scores: lower = more anomalous
if_scores = iso_forest.score_samples(X)
if_labels = iso_forest.predict(X)  # -1 = anomaly, 1 = normal

n_anomalies_if = (if_labels == -1).sum()
print(f"Isolation Forest anomalies: {n_anomalies_if} / {len(df)} ({n_anomalies_if / len(df) * 100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(if_scores, color='steelblue', linewidth=0.5, alpha=0.7)
threshold = np.percentile(if_scores, 5)
ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1, label=f'5th percentile threshold ({threshold:.3f})')
ax.set_title('Isolation Forest — Anomaly Scores (lower = more anomalous)')
ax.set_xlabel('Observation index')
ax.set_ylabel('Anomaly score')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Local Outlier Factor (LOF)

<!-- ANOM-01 -->
LOF compares the local density of a point to its neighbors. Points in low-density regions
relative to their neighbors receive high LOF scores — these are anomalies.

Parameters: `n_neighbors=20`, `contamination='auto'`

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination='auto',
    n_jobs=-1
)
lof_labels = lof.fit_predict(X)
lof_scores = -lof.negative_outlier_factor_  # negate so higher = more anomalous

n_anomalies_lof = (lof_labels == -1).sum()
print(f"LOF anomalies: {n_anomalies_lof} / {len(df)} ({n_anomalies_lof / len(df) * 100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(lof_scores, color='darkorange', linewidth=0.5, alpha=0.7)
threshold_lof = np.percentile(lof_scores, 95)
ax.axhline(y=threshold_lof, color='red', linestyle='--', linewidth=1, label=f'95th percentile threshold ({threshold_lof:.3f})')
ax.set_title('LOF — Outlier Factor Scores (higher = more anomalous)')
ax.set_xlabel('Observation index')
ax.set_ylabel('LOF score')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. DBSCAN Anomaly Flagging

<!-- ANOM-01 -->
DBSCAN groups dense clusters and assigns label -1 to points that do not belong to any cluster.
These noise points are treated as anomalies.

eps is estimated from the k-distance elbow plot below. min_samples is set to
`max(5, int(len(df) * 0.005))` — at least 5, or 0.5% of the dataset.

In [ ]:
from sklearn.neighbors import NearestNeighbors

K = 5
nbrs = NearestNeighbors(n_neighbors=K, n_jobs=-1).fit(X)
distances, _ = nbrs.kneighbors(X)
k_distances = np.sort(distances[:, K - 1])[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_distances, color='teal')
ax.set_title(f'k-Distance Plot (k={K}) — Look for elbow to select eps')
ax.set_xlabel('Points sorted by distance (descending)')
ax.set_ylabel(f'{K}-NN distance')
plt.tight_layout()
plt.show()

# Heuristic: use the distance at the 95th percentile of the sorted k-distance curve
eps_heuristic = float(np.percentile(k_distances, 5))
min_samples = max(5, int(len(df) * 0.005))
print(f"Heuristic eps:   {eps_heuristic:.4f} (95th percentile of sorted k-distances)")
print(f"min_samples:     {min_samples}")
print(f"Inspect the elbow in the plot above. If the elbow is clearly at a different value,")
print(f"adjust eps manually before running the consensus section.")

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=eps_heuristic, min_samples=min_samples, n_jobs=-1)
dbscan_labels = dbscan.fit_predict(X)

n_anomalies_dbscan = (dbscan_labels == -1).sum()
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
print(f"DBSCAN clusters:  {n_clusters}")
print(f"DBSCAN anomalies: {n_anomalies_dbscan} / {len(df)} ({n_anomalies_dbscan / len(df) * 100:.1f}%)")

# Cluster distribution
label_counts = pd.Series(dbscan_labels).value_counts().sort_index()
display(Markdown("**DBSCAN label distribution:**"))
display(label_counts.rename('count').rename_axis('label').reset_index())

## 6. Consensus Anomaly Flag

<!-- D-02: anomaly_flag = 1 if flagged by 2+ algorithms -->
A point is flagged as anomalous if at least 2 out of 3 algorithms agree.
This majority-vote approach reduces false positives from any single algorithm.

In [ ]:
results_df = pd.DataFrame({
    'isolation_forest_score': if_scores,    # lower = more anomalous
    'lof_score': lof_scores,                # higher = more anomalous
    'dbscan_label': dbscan_labels,          # -1 = anomaly
    'if_flag': (if_labels == -1).astype(int),
    'lof_flag': (lof_labels == -1).astype(int),
    'dbscan_flag': (dbscan_labels == -1).astype(int),
})

# Majority vote: anomaly_flag = 1 if flagged by 2+ algorithms
results_df['votes'] = results_df['if_flag'] + results_df['lof_flag'] + results_df['dbscan_flag']
results_df['anomaly_flag'] = (results_df['votes'] >= 2).astype(int)

n_consensus = results_df['anomaly_flag'].sum()
pct_consensus = n_consensus / len(results_df) * 100

print(f"Consensus anomalies (2+ algorithms): {n_consensus} / {len(results_df)} ({pct_consensus:.1f}%)")
print()
print("Algorithm agreement breakdown:")
print(results_df['votes'].value_counts().sort_index().rename('count').rename_axis('algorithms_agreed'))

# Show top anomalous rows
top_anomalies = results_df[results_df['anomaly_flag'] == 1].sort_values('votes', ascending=False).head(10)
if len(top_anomalies) > 0:
    display(Markdown("**Top consensus anomalies (showing index, scores, flag):**"))
    display(top_anomalies[['isolation_forest_score', 'lof_score', 'dbscan_label', 'votes', 'anomaly_flag']].round(4))

## 7. Save Anomaly Flags

<!-- ANOM-04 -->
Writes `data/processed/anomaly_flags_{filename}.csv` with all score columns and the consensus flag.
Row index aligns with the input dataset (rows where all numeric values were null are excluded).

In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

flag_filename = f"anomaly_flags_{input_file.name}"
flag_path = processed_dir / flag_filename

# Final output columns per D-02
output_df = results_df[['isolation_forest_score', 'lof_score', 'dbscan_label', 'anomaly_flag']].copy()
output_df.to_csv(flag_path, index=True, index_label='row_index')

print(f"Anomaly flags written: {flag_path}")
print(f"Rows: {len(output_df):,}")
print(f"Anomalies flagged: {output_df['anomaly_flag'].sum()} ({output_df['anomaly_flag'].mean() * 100:.1f}%)")
print()
print("Columns in flag CSV:")
print("  row_index            — original row index (aligns with input dataset)")
print("  isolation_forest_score — continuous score (lower = more anomalous)")
print("  lof_score            — continuous score (higher = more anomalous)")
print("  dbscan_label         — cluster label (-1 = noise/anomaly)")
print("  anomaly_flag         — binary consensus (1 = flagged by 2+ algorithms)")

## Caveats

<!-- OUT-03: correlation is not causation disclaimer required in all stakeholder reports -->
- **Correlation is not causation** — anomaly scores indicate statistical deviation, not causal defects.
- Only numeric columns were used for scoring. Categorical columns were excluded from all three algorithms.
- DBSCAN eps was estimated heuristically from the k-distance plot — review the plot and adjust if the elbow is unclear.
- Isolation Forest and LOF use `contamination='auto'`, which estimates the contamination fraction from the data. Adjust if domain knowledge suggests a specific expected rate.
- The consensus flag (2+ algorithm agreement) reduces false positives but may miss anomalies that only one algorithm detects. Use the individual score columns for fine-grained filtering.
- Rows where all numeric values were null were excluded from scoring. Check the Data Load section for the row count used.
- The flag CSV in `data/processed/` is suitable for joining back to the original dataset using `row_index`.